# Module 2: Data Augmentation
Expanding the dataset through Sentence Segmentation and Artificial Text Augmentation (Random Swap, Random Deletion).

In [ ]:
import pandas as pd
import nltk
import random
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

df_train = pd.read_csv('data/train.csv')

## 1. Sentence-level Segmentation
Split large paragraphs into individual sentences.

In [ ]:
sentence_data = []
for idx, row in df_train.iterrows():
    sentences = sent_tokenize(str(row['text']))
    for s in sentences:
        if len(s.strip()) > 10:
            sentence_data.append(s.strip())

print(f"Original documents: {len(df_train)}, Extracted sentences: {len(sentence_data)}")

## 2. Artificial Data Augmentation (EDA)
To solve the issue of a small dataset, we apply Easy Data Augmentation techniques:
- **Random Swap**: Swapping two adjacent or random words.
- **Random Deletion**: Deleting a random word to simulate noise.
This inflates the dataset size 5x times.

In [ ]:
def augment_sentence(sentence, num_augmented=5):
    words = sentence.split()
    if len(words) < 5:
        return []
    
    augmented = set()
    for _ in range(num_augmented):
        aug_words = words.copy()
        
        if random.random() > 0.5:
            # Random Swap (swap 2 words)
            idx1, idx2 = random.sample(range(len(aug_words)), 2)
            aug_words[idx1], aug_words[idx2] = aug_words[idx2], aug_words[idx1]
        else:
            # Random Deletion (remove 1 word)
            del_idx = random.randint(0, len(aug_words)-1)
            aug_words.pop(del_idx)
            
        augmented.add(" ".join(aug_words))
        
    return list(augmented)

random.seed(42)
augmented_data = []

# Apply augmentation
for s in sentence_data:
    augmented_data.append(s) # keep original
    augmented_data.extend(augment_sentence(s, num_augmented=5))

# Remove duplicates overall just in case
augmented_data = list(set(augmented_data))

df_sentences = pd.DataFrame({'sentence': augmented_data})
df_sentences.to_csv('data/train_sentences.csv', index=False)
print(f"Final Augmented Dataset Size: {len(df_sentences)} sentences!")

## 3. Incremental sequences for LSTM
Convert sentences to sliding window numerical data.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import string

corpus = df_sentences['sentence'].tolist()
cleaned_corpus = [str(c).lower().translate(str.maketrans('', '', string.punctuation)) for c in corpus]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(cleaned_corpus)
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in cleaned_corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

X, y = input_sequences[:,:-1], input_sequences[:,-1]
print(f"Created {len(X)} incremental sequences for LSTM training.")

np.save('data/lstm_X.npy', X)
np.save('data/lstm_y.npy', y)

import pickle
with open('data/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)